<a href="https://colab.research.google.com/github/Adamphoenix003/GNN-LinkPrediction/blob/main/SEAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.2 MB/s eta 0:00:00


In [3]:
import torch
import numpy as np
import pandas as pd

content_path = "/content/sample_data/cora.content"
# Load content file
content = pd.read_csv(content_path, sep="\t", header=None)

paper_ids = content[0].values
features = content.iloc[:, 1:-1].values
labels_raw = content.iloc[:, -1].values

# Map paper IDs to indices
id_map = {id_: i for i, id_ in enumerate(paper_ids)}

# Encode labels
label_map = {label: i for i, label in enumerate(set(labels_raw))}
labels = np.array([label_map[l] for l in labels_raw])

x = torch.tensor(features, dtype=torch.float)
y = torch.tensor(labels, dtype=torch.long)

In [4]:
cites_path = "/content/sample_data/cora.cites"

cites = pd.read_csv(cites_path, sep="\t", header=None)

edges = []

for _, row in cites.iterrows():
    src, dst = row[0], row[1]
    if src in id_map and dst in id_map:
        edges.append([id_map[src], id_map[dst]])

edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

# Make undirected
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

print("Nodes:", x.shape[0])
print("Edges:", edge_index.shape[1])


Nodes: 2708
Edges: 10858


In [5]:
from torch_geometric.data import Data

data = Data(x=x, edge_index=edge_index, y=y)

In [6]:
from torch_geometric.transforms import RandomLinkSplit

transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    split_labels=True
)

train_data, val_data, test_data = transform(data)

In [7]:
import networkx as nx

def drnl_node_labeling(sub_edge_index, src, dst, num_nodes):

    G = nx.Graph()
    G.add_nodes_from(range(num_nodes))
    G.add_edges_from(sub_edge_index.t().tolist())

    dist_src = nx.single_source_shortest_path_length(G, src)
    dist_dst = nx.single_source_shortest_path_length(G, dst)

    labels = []

    for i in range(num_nodes):
        d1 = dist_src.get(i, 100)
        d2 = dist_dst.get(i, 100)

        if i == src or i == dst:
            labels.append(1)
        elif d1 == 100 or d2 == 100:
            labels.append(0)
        else:
            labels.append(1 + min(d1, d2))

    return torch.tensor(labels)

In [8]:
from torch_geometric.utils import k_hop_subgraph

def extract_subgraph(src, dst, edge_index, x, hops=2):

    subset, sub_edge_index, mapping, _ = k_hop_subgraph(
        [src, dst],
        hops,
        edge_index,
        relabel_nodes=True
    )

    sub_x = x[subset]

    src_new = mapping[0].item()
    dst_new = mapping[1].item()

    # 🔥 Remove target edge
    mask = ~(
        ((sub_edge_index[0] == src_new) & (sub_edge_index[1] == dst_new)) |
        ((sub_edge_index[0] == dst_new) & (sub_edge_index[1] == src_new))
    )

    sub_edge_index = sub_edge_index[:, mask]

    sub_z = drnl_node_labeling(
        sub_edge_index,
        src_new,
        dst_new,
        sub_x.size(0)
    )

    return sub_x, sub_edge_index, sub_z

In [9]:
from tqdm import tqdm

def build_seal_dataset(split_data):

    pos_edges = split_data.pos_edge_label_index.t()
    neg_edges = split_data.neg_edge_label_index.t()

    graphs = []
    labels = []

    print("Extracting subgraphs...")

    for edge in tqdm(pos_edges):
        src, dst = edge.tolist()
        graphs.append(extract_subgraph(
            src, dst,
            train_data.edge_index,
            data.x
        ))
        labels.append(1)

    for edge in tqdm(neg_edges):
        src, dst = edge.tolist()
        graphs.append(extract_subgraph(
            src, dst,
            train_data.edge_index,
            data.x
        ))
        labels.append(0)

    return graphs, torch.tensor(labels)

In [10]:
train_graphs, train_labels = build_seal_dataset(train_data)
val_graphs, val_labels = build_seal_dataset(val_data)
test_graphs, test_labels = build_seal_dataset(test_data)

Extracting subgraphs...


100%|██████████| 4345/4345 [00:08<00:00, 536.85it/s]


Extracting subgraphs...


100%|██████████| 542/542 [00:00<00:00, 583.47it/s]


Extracting subgraphs...


100%|██████████| 542/542 [00:00<00:00, 588.22it/s]


In [11]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class SEAL_SortPooling(nn.Module):
    def __init__(self, in_channels, hidden_channels, k):
        super().__init__()

        self.k = k

        self.conv1 = GCNConv(in_channels + 1, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)

        self.lin1 = nn.Linear(hidden_channels * k, hidden_channels)
        self.lin2 = nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index, z):

        z = z.unsqueeze(1).float()
        x = torch.cat([x, z], dim=1)

        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        # Sort by last embedding dimension
        sort_idx = torch.argsort(x[:, -1], descending=True)
        x = x[sort_idx]

        if x.size(0) < self.k:
            pad = torch.zeros(self.k - x.size(0), x.size(1)).to(x.device)
            x = torch.cat([x, pad], dim=0)
        else:
            x = x[:self.k]

        x = x.view(1, -1)

        x = F.relu(self.lin1(x))
        x = torch.sigmoid(self.lin2(x))

        return x.view(-1)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SEAL_SortPooling(x.shape[1], 128, k=30).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

def train():
    model.train()
    total_loss = 0

    for (sub_x, sub_edge_index, sub_z), label in zip(train_graphs, train_labels):

        sub_x = sub_x.to(device)
        sub_edge_index = sub_edge_index.to(device)
        sub_z = sub_z.to(device)
        label = label.float().unsqueeze(0).to(device)

        optimizer.zero_grad()

        out = model(sub_x, sub_edge_index, sub_z)
        loss = criterion(out, label)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_graphs)

for epoch in range(1, 31):
    loss = train()
    print(f"Epoch {epoch}, Loss: {loss:.4f}")

Epoch 1, Loss: 16.0119
Epoch 2, Loss: 6.3818
Epoch 3, Loss: 5.1000
Epoch 4, Loss: 6.2359
Epoch 5, Loss: 5.5786
Epoch 6, Loss: 5.0872
Epoch 7, Loss: 4.9191
Epoch 8, Loss: 5.0490
Epoch 9, Loss: 1.0888
Epoch 10, Loss: 4.9179
Epoch 11, Loss: 4.9598
Epoch 12, Loss: 4.9289
Epoch 13, Loss: 4.9005
Epoch 14, Loss: 3.3537
Epoch 15, Loss: 5.7022
Epoch 16, Loss: 3.4073
Epoch 17, Loss: 5.6962
Epoch 18, Loss: 3.4815
Epoch 19, Loss: 3.4662
Epoch 20, Loss: 1.1178
Epoch 21, Loss: 1.1135
Epoch 22, Loss: 0.2638
Epoch 23, Loss: 0.1375
Epoch 24, Loss: 1.1152
Epoch 25, Loss: 0.2484
Epoch 26, Loss: 1.0626
Epoch 27, Loss: 0.2329
Epoch 28, Loss: 0.4546
Epoch 29, Loss: 0.2151
Epoch 30, Loss: 0.1324


In [14]:
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

@torch.no_grad()
def evaluate(graphs, labels):

    model.eval()
    preds = []
    true = []

    for (sub_x, sub_edge_index, sub_z), label in zip(graphs, labels):

        sub_x = sub_x.to(device)
        sub_edge_index = sub_edge_index.to(device)
        sub_z = sub_z.to(device)

        out = model(sub_x, sub_edge_index, sub_z)

        preds.append(out.item())
        true.append(label.item())

    auc = roc_auc_score(true, preds)
    ap = average_precision_score(true, preds)

    return auc, ap


val_auc, val_ap = evaluate(val_graphs, val_labels)
test_auc, test_ap = evaluate(test_graphs, test_labels)

print("Validation AUC:", val_auc)
print("Validation AP :", val_ap)
print("Test AUC:", test_auc)
print("Test AP :", test_ap)

Validation AUC: 0.7246905679388898
Validation AP : 0.7191594957747327
Test AUC: 0.7154416470363965
Test AP : 0.7112741526364721


In [15]:
import torch
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

content_path = "/content/sample_data/cora.content"
content = pd.read_csv(content_path, sep="\t", header=None)

paper_ids = content[0].values
features = content.iloc[:, 1:-1].values
labels_raw = content.iloc[:, -1].values

# Normalize features
scaler = StandardScaler()
features = scaler.fit_transform(features)

# Map paper IDs
id_map = {id_: i for i, id_ in enumerate(paper_ids)}

# Encode labels
label_map = {label: i for i, label in enumerate(set(labels_raw))}
labels = np.array([label_map[l] for l in labels_raw])

x = torch.tensor(features, dtype=torch.float)
y = torch.tensor(labels, dtype=torch.long)

In [16]:
def extract_subgraph(src, dst, edge_index, x, hops=3):

    subset, sub_edge_index, mapping, _ = k_hop_subgraph(
        [src, dst],
        hops,
        edge_index,
        relabel_nodes=True
    )

    sub_x = x[subset]

    src_new = mapping[0].item()
    dst_new = mapping[1].item()

    mask = ~(
        ((sub_edge_index[0] == src_new) & (sub_edge_index[1] == dst_new)) |
        ((sub_edge_index[0] == dst_new) & (sub_edge_index[1] == src_new))
    )

    sub_edge_index = sub_edge_index[:, mask]

    sub_z = drnl_node_labeling(
        sub_edge_index,
        src_new,
        dst_new,
        sub_x.size(0)
    )

    return sub_x, sub_edge_index, sub_z

In [17]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class SEAL_SortPooling(nn.Module):

    def __init__(self, in_channels, hidden_channels, k):
        super().__init__()

        self.k = k

        self.conv1 = GCNConv(in_channels + 1, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, hidden_channels)

        self.lin1 = nn.Linear(hidden_channels * k, hidden_channels)
        self.lin2 = nn.Linear(hidden_channels, 1)

        self.dropout = 0.5

    def forward(self, x, edge_index, z):

        z = z.unsqueeze(1).float()
        x = torch.cat([x, z], dim=1)

        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = F.relu(self.conv3(x, edge_index))

        # SortPooling
        sort_idx = torch.argsort(x[:, -1], descending=True)
        x = x[sort_idx]

        if x.size(0) < self.k:
            pad = torch.zeros(self.k - x.size(0), x.size(1)).to(x.device)
            x = torch.cat([x, pad], dim=0)
        else:
            x = x[:self.k]

        x = x.view(1, -1)

        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = torch.sigmoid(self.lin2(x))

        return x.view(-1)

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SEAL_SortPooling(x.shape[1], 128, k=60).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

In [19]:

for epoch in range(1, 31):

    loss = train()

    if epoch % 5 == 0:
        val_auc, val_ap = evaluate(val_graphs, val_labels)

        print(f"Epoch {epoch}")
        print(f"Loss: {loss:.4f}")
        print(f"Val AUC: {val_auc:.4f}")
        print(f"Val AP: {val_ap:.4f}")
        print()

Epoch 5
Loss: 6.6943
Val AUC: 0.4834
Val AP: 0.5169

Epoch 10
Loss: 6.5835
Val AUC: 0.4492
Val AP: 0.5003

Epoch 15
Loss: 2.0673
Val AUC: 0.5069
Val AP: 0.5134

Epoch 20
Loss: 3.0117
Val AUC: 0.6977
Val AP: 0.6531

Epoch 25
Loss: 1.3136
Val AUC: 0.7496
Val AP: 0.7078

Epoch 30
Loss: 1.4498
Val AUC: 0.7347
Val AP: 0.7044



In [20]:
val_auc, val_ap = evaluate(val_graphs, val_labels)
test_auc, test_ap = evaluate(test_graphs, test_labels)

print("Validation AUC:", val_auc)
print("Validation AP :", val_ap)
print("Test AUC:", test_auc)
print("Test AP :", test_ap)

Validation AUC: 0.7346509442954209
Validation AP : 0.704422300857704
Test AUC: 0.7178177040072984
Test AP : 0.6930800330489146
